# Synthetic monitoring for a checkout flow: From 20 alerts to 4

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/synthetic-monitoring-checkout-alerts-slo).
It builds the lab, replays the week (about three hours) and counts what each monitor, rule and SLO saw.
The last section cleans up.

Requires:

- Elastic Cloud Hosted 9.5 or later, with the `us_east` and `germany` managed locations and a subscription that includes maintenance windows.
- Bean Box from `app/` on a public host, with an HTTPS URL and a public TCP port (`PORT=8080 GATEWAY_PORT=7070 CHAOS_TOKEN=<random> npm start`).
- Docker and Node.js 22 or 24 on this machine. The app simulates the office network by blocking this machine's public IP, so run the notebook where the Private Location agent runs.
- A `.env` copied from `.env.example`, with an API key created by a superuser with no role descriptors:

```
POST /_security/api_key
{ "name": "synthetic-monitoring-checkout-notebook", "role_descriptors": {} }
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

The IDs this notebook creates are saved to `runs/lab-state.json`, so **Clean up** works from a fresh kernel after running this cell.

In [ ]:
import json
import os
import pathlib
import socket
import subprocess
import time
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
SYNTHETICS_API_KEY = os.getenv("SYNTHETICS_API_KEY") or API_KEY

SHOP_URL = os.environ["SHOP_URL"].rstrip("/")
SHOP_HOST = SHOP_URL.split("://", 1)[1].split("/", 1)[0]
GATEWAY_ADDRESS = os.environ["GATEWAY_ADDRESS"]  # host:port of the public TCP proxy
CHAOS_TOKEN = os.environ["CHAOS_TOKEN"]
LEAD_MINUTES = int(os.getenv("LEAD_MINUTES", "10"))

# Names of everything this notebook creates. Clean up deletes exactly these.
PROJECT_ID = "bean-box-checkout"
BASELINE_PROJECT_ID = "bean-box-checkout-baseline"
LOCATION_LABEL = "office-network"
AGENT_POLICY_NAME = "bean-box office-network"
AGENT_CONTAINER = "bean-box-office-agent"
RULE_NAME = "Checkout monitors down"
SLO_NAME = "Checkout journey availability"
MW_TITLE = "Payment provider maintenance (bean-box)"

HERE = pathlib.Path.cwd()
RUNS = HERE / "runs"
RUNS.mkdir(exist_ok=True)
PROJECT_DIR = RUNS / "project"
BASELINE_DIR = RUNS / "project-baseline"
STATE_FILE = RUNS / "lab-state.json"
state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state(**values):
    state.update(values)
    STATE_FILE.write_text(json.dumps(state, indent=2, default=str))


es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def kbn(method, path, internal=False, ok=(), **kwargs):
    """Call a Kibana API in the configured space. Statuses in `ok` return None instead of raising."""
    headers = dict(KBN_HEADERS)
    if internal:
        headers["x-elastic-internal-origin"] = "kibana"
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=headers, timeout=120, **kwargs
    )
    if response.status_code in ok:
        return None
    if response.status_code >= 400:
        raise RuntimeError(
            f"{method} {path} -> {response.status_code}: {response.text[:600]}"
        )
    return response.json() if response.text else {}


def esql(query):
    """Run an ES|QL query and return the rows as dicts."""
    response = es.esql.query(query=query)
    columns = [c["name"] for c in response["columns"]]
    return [dict(zip(columns, row)) for row in response["values"]]


def table(rows, columns):
    """Print rows as a fixed-width text table."""
    widths = {
        c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) if rows else len(c)
        for c in columns
    }
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))


def iso(ts):
    return ts.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


CHAOS_HEADERS = {"Authorization": f"Bearer {CHAOS_TOKEN}"}
CLEAR = {
    "error_rate": 0,
    "latency_ms": 0,
    "gateway_latency_ms": 0,
    "gateway_down": False,
    "broken_asset": False,
    "health_down": False,
    "block_ips": [],
}


def chaos(change, note=""):
    """Change the failure injection state of Bean Box through its /__chaos endpoint."""
    for attempt in range(5):
        try:
            response = requests.post(
                f"{SHOP_URL}/__chaos",
                json={**change, "note": note},
                headers=CHAOS_HEADERS,
                timeout=15,
            )
            response.raise_for_status()
            return response.json()["chaos"]
        except (
            requests.RequestException
        ) as error:  # keep going on a transient network error
            print(f"  retry {attempt}: {error}")
            time.sleep(5)
    raise RuntimeError(f"could not apply chaos change: {note}")


STACK_VERSION = es.info()["version"]["number"]
print(f"elasticsearch {STACK_VERSION} | space={SPACE} | state keys: {sorted(state)}")

## 2. Check the Bean Box app

Checks `/health`, the gateway's `PING`/`PONG`, and the public IP the app sees for this machine, which the office network events block.

In [ ]:
health = requests.get(f"{SHOP_URL}/health", timeout=10).json()
print("health:", health["status"])

gateway_host, gateway_port = GATEWAY_ADDRESS.rsplit(":", 1)
with socket.create_connection((gateway_host, int(gateway_port)), timeout=10) as sock:
    sock.sendall(b"PING\n")
    print("gateway:", sock.recv(64).decode().strip())

OFFICE_IP = os.getenv("OFFICE_IP")
if not OFFICE_IP:
    whoami = requests.get(f"{SHOP_URL}/__whoami", headers=CHAOS_HEADERS, timeout=10)
    if whoami.status_code == 404:
        raise RuntimeError(
            "This Bean Box build has no /__whoami. Redeploy app/ or set OFFICE_IP in .env."
        )
    whoami.raise_for_status()
    OFFICE_IP = whoami.json()["ip"]
save_state(office_ip_found=True)
print(
    "office IP the app will block:",
    OFFICE_IP.rsplit(".", 1)[0] + ".x" if "." in OFFICE_IP else "(IPv6)",
)
print("chaos state:", chaos(CLEAR, "notebook: start from a clean state"))

## 3. Create the office-network Private Location

Browser monitors need the `elastic-agent-complete` image at the Stack version. The first pull takes a few minutes.

In [ ]:
policies = kbn(
    "GET",
    "/api/fleet/agent_policies",
    params={"kuery": f'ingest-agent-policies.name:"{AGENT_POLICY_NAME}"'},
)["items"]
if policies:
    policy = policies[0]
else:
    policy = kbn(
        "POST",
        "/api/fleet/agent_policies",
        json={
            "name": AGENT_POLICY_NAME,
            "namespace": "default",
            "description": "Private Location for the synthetic monitoring checkout lab",
            "monitoring_enabled": ["logs", "metrics"],
        },
    )["item"]
save_state(agent_policy_id=policy["id"])

locations = kbn("GET", "/api/synthetics/private_locations")
location = next((l for l in locations if l["label"] == LOCATION_LABEL), None)
if location and location["agentPolicyId"] != policy["id"]:
    raise RuntimeError(
        f"A Private Location called {LOCATION_LABEL} already uses another agent policy."
    )
if not location:
    location = kbn(
        "POST",
        "/api/synthetics/private_locations",
        json={"label": LOCATION_LABEL, "agentPolicyId": policy["id"]},
    )
save_state(private_location_id=location["id"])
print(f"policy {policy['id']} | location {location['label']} ({location['id']})")

# Filter client-side: the kuery prefix for enrollment keys changed across versions.
listed = kbn("GET", "/api/fleet/enrollment_api_keys", params={"perPage": 1000})
enrollment_token = next(
    k["api_key"]
    for k in listed.get("items") or listed.get("list", [])
    if k.get("policy_id") == policy["id"] and k.get("active", True)
)
fleet_url = (
    os.getenv("FLEET_URL")
    or next(
        h
        for h in kbn("GET", "/api/fleet/fleet_server_hosts")["items"]
        if h.get("is_default")
    )["host_urls"][0]
)

running = subprocess.run(
    ["docker", "ps", "-q", "-f", f"name=^{AGENT_CONTAINER}$"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
if not running:
    subprocess.run(["docker", "rm", "-f", AGENT_CONTAINER], capture_output=True)
    started = subprocess.run(
        [
            "docker",
            "run",
            "-d",
            "--name",
            AGENT_CONTAINER,
            "--env",
            "FLEET_ENROLL=1",
            "--env",
            f"FLEET_URL={fleet_url}",
            "--env",
            f"FLEET_ENROLLMENT_TOKEN={enrollment_token}",
            "--cap-add=NET_RAW",
            "--cap-add=SETUID",
            f"docker.elastic.co/elastic-agent/elastic-agent-complete:{STACK_VERSION}",
        ],
        capture_output=True,
        text=True,
    )
    if started.returncode:
        raise RuntimeError(started.stderr[-1000:])
save_state(agent_container=AGENT_CONTAINER)

deadline = time.time() + 900
while True:
    agents = kbn(
        "GET",
        "/api/fleet/agents",
        params={"kuery": f'policy_id:"{policy["id"]}"', "perPage": 100},
    )["items"]
    statuses = [a.get("status") or a.get("last_checkin_status") for a in agents]
    if "online" in statuses:
        break
    if time.time() > deadline:
        raise TimeoutError(f"agent not online after 15 minutes: {statuses}")
    time.sleep(15)
print(f"agent online: {AGENT_CONTAINER} (elastic-agent-complete:{STACK_VERSION})")

## 4. Push the Synthetics project (Steps 1 to 3)

Copies `synthetics/` to `runs/project/` with your shop URL, gateway address and space, runs the journey once locally and pushes it.

In [ ]:
SRC = HERE / "synthetics"
FILES = [
    "package.json",
    "package-lock.json",
    "synthetics.config.ts",
    "lightweight/checkout.yml",
    "journeys/checkout.journey.ts",
]


def render_project(dest, project_id, maintenance_window_id=None, baseline=False):
    """Copy synthetics/ to dest with this lab's URLs. The baseline copy uses the init defaults."""
    for rel in FILES:
        text = (SRC / rel).read_text()
        text = (
            text.replace("https://bean-box-production-0ad8.up.railway.app", SHOP_URL)
            .replace("metro.proxy.rlwy.net:35021", GATEWAY_ADDRESS)
            .replace("bean-box-production-0ad8.up.railway.app", SHOP_HOST)
        )
        if rel == "synthetics.config.ts":
            text = text.replace("id: 'bean-box-checkout'", f"id: '{project_id}'")
            text = text.replace("space: 'default'", f"space: '{SPACE}'")
            if maintenance_window_id:
                assert "      alert: {" in text
                text = text.replace(
                    "      alert: {",
                    f"      maintenanceWindows: ['{maintenance_window_id}'],\n      alert: {{",
                    1,
                )
            if baseline:
                text = text.replace(
                    "status: { enabled: false }", "status: { enabled: true }"
                )
                text = text.replace("schedule: 5,", "schedule: 10,")
        if baseline and rel == "journeys/checkout.journey.ts":
            text = text.replace(
                "journey('Checkout journey'", "journey('Checkout journey (baseline)'"
            )
            text = text.replace("schedule: 5 }", "schedule: 10 }")
        if baseline and rel == "lightweight/checkout.yml":
            text = text.replace("'@every 1m'", "'@every 3m'")
            for name in [
                "Checkout health (HTTP)",
                "Payment gateway (TCP)",
                "Shop edge (ICMP)",
            ]:
                text = text.replace(
                    f"name: {name}", f"name: {name.replace(')', ', baseline)')}"
                )
        target = dest / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(text)


CLI_ENV = {
    **os.environ,
    "KIBANA_URL": KIBANA_URL,
    "SYNTHETICS_API_KEY": SYNTHETICS_API_KEY,
}


def run(cmd, cwd, tail=25):
    """Run a command, print the end of its output, raise if it fails."""
    done = subprocess.run(cmd, cwd=cwd, env=CLI_ENV, capture_output=True, text=True)
    output = (done.stdout + done.stderr).strip().splitlines()
    print("\n".join(output[-tail:]))
    if done.returncode:
        raise RuntimeError(f"{' '.join(cmd)} exited with {done.returncode}")


def project_monitors(project_id):
    found = kbn(
        "GET",
        "/api/synthetics/monitors",
        params={"projects": project_id, "perPage": 100},
    )
    return found.get("monitors", [])


render_project(PROJECT_DIR, PROJECT_ID)
run(["npm", "ci", "--no-audit", "--no-fund"], PROJECT_DIR, tail=5)
run(["npx", "@elastic/synthetics", "journeys"], PROJECT_DIR)
run(["npx", "@elastic/synthetics", "push", "--yes"], PROJECT_DIR)
save_state(projects=[PROJECT_ID])


def describe(m):
    schedule = m.get("schedule") or {}
    locations = [l.get("label") or l.get("id") for l in m.get("locations", [])]
    return f"{m['name']:<36} {m.get('type', ''):<8} every {schedule.get('number')}{schedule.get('unit')}  {locations}"


for m in project_monitors(PROJECT_ID):
    print(describe(m))

## 5. Push the baseline copy for the default status rule

A second copy of the project with the `init` schedules and alerting on, to measure the default rule.
`push` doesn't create the default rule, so the cell creates it as the Synthetics **Alerts** menu does. Clean up deletes it only if this cell created it.

In [ ]:
render_project(BASELINE_DIR, BASELINE_PROJECT_ID, baseline=True)
if not (BASELINE_DIR / "node_modules").exists():
    (BASELINE_DIR / "node_modules").symlink_to(PROJECT_DIR / "node_modules")
run(["npx", "@elastic/synthetics", "push", "--yes"], BASELINE_DIR)
save_state(projects=[PROJECT_ID, BASELINE_PROJECT_ID])


def find_rules(kql):
    return kbn(
        "GET", "/api/alerting/rules/_find", params={"filter": kql, "per_page": 100}
    )["data"]


DEFAULT_RULE_KQL = (
    'alert.attributes.alertTypeId:"xpack.synthetics.alerts.monitorStatus" '
    'and alert.attributes.tags:"SYNTHETICS_DEFAULT_ALERT"'
)
default_rules = find_rules(DEFAULT_RULE_KQL)
if not default_rules:
    kbn("POST", "/internal/synthetics/enable_default_alerting", internal=True)
    default_rules = find_rules(DEFAULT_RULE_KQL)
    save_state(default_rule_created=True)
if not default_rules:
    raise RuntimeError(
        "No default status rule. Open Synthetics > Alerts once in Kibana, then rerun this cell."
    )
default_rule = default_rules[0]
save_state(default_rule_id=default_rule["id"])
print(
    f"default rule: {default_rule['name']} ({default_rule['id']}) enabled={default_rule['enabled']}"
)
for m in project_monitors(BASELINE_PROJECT_ID):
    print(describe(m))

## 6. Add the maintenance window and the scoped status rule (Step 4)

The window is daily recurring because a one-off window stops the monitors in 9.5.6 ([kibana#297010](https://github.com/elastic/kibana/issues/297010)).
Rerunning this cell picks a new start time for the week.

In [ ]:
T0 = (datetime.now(timezone.utc) + timedelta(minutes=LEAD_MINUTES)).replace(
    second=0, microsecond=0
)
if state.get("maintenance_window_id"):
    kbn(
        "DELETE", f"/api/maintenance_window/{state['maintenance_window_id']}", ok=(404,)
    )
maintenance_window = kbn(
    "POST",
    "/api/maintenance_window",
    json={
        "title": MW_TITLE,
        "enabled": True,
        "schedule": {
            "custom": {
                # This API wants milliseconds in the start date (YYYY-MM-DDTHH:mm:ss.sssZ).
                "start": (T0 + timedelta(minutes=60)).strftime(
                    "%Y-%m-%dT%H:%M:%S.000Z"
                ),
                "duration": "5m",
                "recurring": {"every": "1d"},
                "timezone": "UTC",
            }
        },
    },
)
save_state(maintenance_window_id=maintenance_window["id"], t0=T0.isoformat())
print(
    f"T0 = {iso(T0)} | maintenance window {maintenance_window['id']} "
    f"{iso(T0 + timedelta(minutes=60))} for 5m, daily"
)

render_project(PROJECT_DIR, PROJECT_ID, maintenance_window_id=maintenance_window["id"])
run(["npx", "@elastic/synthetics", "push", "--yes"], PROJECT_DIR)

rule_body = json.loads((HERE / "rules" / "checkout-status-rule.json").read_text())
existing = find_rules(f'alert.attributes.name:"{RULE_NAME}"')
rule = existing[0] if existing else kbn("POST", "/api/alerting/rule", json=rule_body)
save_state(rule_id=rule["id"])
print(
    f"rule: {rule['name']} ({rule['id']}) params={json.dumps(rule['params']['condition'])}"
)

## 7. Create the checkout availability SLO (Step 5)

In [ ]:
JOURNEY_MONITOR_ID = f"checkout-journey-{PROJECT_ID}-{SPACE}"
slo_body = {
    "name": SLO_NAME,
    "description": "Share of final checkout journey attempts that complete an order, per monitor and location",
    "indicator": {
        "type": "sli.synthetics.availability",
        "params": {
            "index": "synthetics-*",
            "monitorIds": [{"value": JOURNEY_MONITOR_ID, "label": "Checkout journey"}],
            "projects": [{"value": PROJECT_ID, "label": PROJECT_ID}],
            "tags": [{"value": "*", "label": "All"}],
            "filter": "",
        },
    },
    "timeWindow": {"duration": "7d", "type": "rolling"},
    "budgetingMethod": "occurrences",
    "objective": {"target": 0.99},
    "groupBy": ["monitor.name", "observer.geo.name", "monitor.id"],
    "tags": ["checkout"],
}
found = kbn(
    "GET", "/api/observability/slos", params={"kqlQuery": f'slo.name:"{SLO_NAME}"'}
)["results"]
slo_id = (
    found[0]["id"]
    if found
    else kbn("POST", "/api/observability/slos", json=slo_body)["id"]
)
save_state(slo_id=slo_id)
print(f"slo: {SLO_NAME} ({slo_id})")

## 8. Replay the week

Injects the article's week through the app's `/__chaos` endpoint, with 25-minute days and 5% background 503s. The cell blocks for about three hours.

In [ ]:
T0 = datetime.fromisoformat(state["t0"])
if datetime.now(timezone.utc) > T0:
    raise RuntimeError(
        "T0 already passed. Rerun section 6 to pick a new T0 and move the maintenance window."
    )

BACKGROUND = {"error_rate": 0.05}
# (minute offset, label, chaos change, ground truth)
TIMELINE = [
    (0, "mon: week starts, background 5% 503s", {**CLEAR, **BACKGROUND}, "noise"),
    (
        35,
        "tue: office network path broken",
        {"block_ips": [OFFICE_IP]},
        "noise:single-location",
    ),
    (41, "tue: office network path restored", {"block_ips": []}, ""),
    (60, "wed: payment provider maintenance starts", {"gateway_down": True}, "planned"),
    (65, "wed: payment provider maintenance ends", {"gateway_down": False}, ""),
    (
        85,
        "thu: payment gateway outage starts",
        {"gateway_down": True},
        "incident:payments",
    ),
    (95, "thu: payment gateway outage ends", {"gateway_down": False}, ""),
    (110, "fri: checkout.js broken", {"broken_asset": True}, "incident:checkout-js"),
    (125, "fri: checkout.js fixed", {"broken_asset": False}, ""),
    (
        130,
        "sat: office network flap 1",
        {"block_ips": [OFFICE_IP]},
        "noise:single-location",
    ),
    (133, "sat: office network flap 1 ends", {"block_ips": []}, ""),
    (
        140,
        "sat: office network flap 2",
        {"block_ips": [OFFICE_IP]},
        "noise:single-location",
    ),
    (143, "sat: office network flap 2 ends", {"block_ips": []}, ""),
    (175, "sun: week ends, all chaos cleared", CLEAR, ""),
]

events = []
for minute, label, change, truth in TIMELINE:
    wait = (T0 + timedelta(minutes=minute) - datetime.now(timezone.utc)).total_seconds()
    if wait > 0:
        time.sleep(wait)
    chaos(change, label)
    now = datetime.now(timezone.utc)
    events.append(
        {"ts": now.isoformat(), "minute": minute, "event": label, "truth": truth}
    )
    save_state(events=events)
    print(f"{now:%H:%M:%S}  +{minute:>3} min  {label}", flush=True)

time.sleep(180)  # let both rules evaluate the last checks
print("week replayed")

## 9. Which monitor caught each event

`final_down` counts only the checks that stayed down after a retest.

In [ ]:
events = state["events"]
T0 = datetime.fromisoformat(events[0]["ts"])
END = datetime.fromisoformat(events[-1]["ts"])

# Event windows from the start/end pairs in the log.
windows, open_events = [], {}
for e in events:
    day = e["event"].split(":")[0]
    ts = datetime.fromisoformat(e["ts"])
    if e["truth"] and e["truth"] != "noise":
        open_events[day] = (
            e["event"].split(": ", 1)[1].replace(" starts", ""),
            e["truth"],
            ts,
        )
    elif day in open_events and not e["truth"]:
        name, truth, start = open_events.pop(day)
        windows.append(
            {"day": day, "event": name, "truth": truth, "start": start, "end": ts}
        )
for w in windows:
    print(
        f"{w['day']}  {w['start']:%H:%M}-{w['end']:%H:%M} UTC  {w['event']} [{w['truth']}]"
    )


def checks_between(start, end, project=PROJECT_ID):
    return esql(
        f"""
FROM synthetics-*
| WHERE monitor.project.id == "{project}" AND summary.down IS NOT NULL
    AND @timestamp >= TO_DATETIME("{iso(start)}") AND @timestamp < TO_DATETIME("{iso(end)}")
| STATS checks = COUNT(*),
        down_attempts = COUNT(CASE(monitor.status == "down", 1, null)),
        final_down = COUNT(CASE(monitor.status == "down" AND summary.final_attempt == true, 1, null))
    BY monitor.name, observer.geo.name
| SORT monitor.name, observer.geo.name
"""
    )


for label, days in [
    ("Thursday gateway outage", ["thu"]),
    ("Friday checkout.js failure", ["fri"]),
    ("Office network path (Tue + Sat)", ["tue", "sat"]),
]:
    rows = []
    for w in (w for w in windows if w["day"] in days):
        rows += checks_between(w["start"], w["end"] + timedelta(minutes=1))
    merged = {}
    for r in rows:
        key = (r["monitor.name"], r["observer.geo.name"])
        m = merged.setdefault(
            key,
            {
                "monitor": key[0],
                "location": key[1],
                "checks": 0,
                "down_attempts": 0,
                "final_down": 0,
            },
        )
        for k in ("checks", "down_attempts", "final_down"):
            m[k] += r[k]
    print(f"\n{label}")
    table(
        list(merged.values()),
        ["monitor", "location", "checks", "down_attempts", "final_down"],
    )
    for monitor in sorted({m["monitor"] for m in merged.values()}):
        locs = [m for m in merged.values() if m["monitor"] == monitor]
        down = [m["location"] for m in locs if m["final_down"]]
        retested = any(m["down_attempts"] for m in locs) and not down
        verdict = (
            f"Down in {len(down)} location(s): {down}"
            if down
            else ("Up after retest" if retested else "Up")
        )
        print(f"  -> {monitor}: {verdict}")

## 10. Count the alerts each rule opened

Each alert goes to the latest event that started before it and ended at most 10 minutes earlier; the rest are background 503s.
Those 503s are random, so the default rule's count changes from run to run.

In [ ]:
CAUSES = {
    "incident:payments": "Thursday gateway outage",
    "incident:checkout-js": "Friday checkout.js failure",
    "planned": "Wednesday planned maintenance",
    "noise:single-location": "Office network path",
    None: "Background 503s",
}
RULES = {state["default_rule_id"]: "default", state["rule_id"]: "scoped"}

alerts = esql(
    f"""
FROM .alerts-observability.uptime.alerts-*
| WHERE kibana.alert.rule.rule_type_id == "xpack.synthetics.alerts.monitorStatus"
    AND kibana.alert.start >= TO_DATETIME("{iso(T0)}")
    AND kibana.alert.start <= TO_DATETIME("{iso(END + timedelta(minutes=5))}")
| STATS started = MIN(kibana.alert.start), locations = VALUES(observer.geo.name)
    BY kibana.alert.uuid, kibana.alert.rule.uuid, monitor.name
| SORT started
| LIMIT 1000
"""
)

counts = {rule: {cause: [] for cause in CAUSES.values()} for rule in RULES.values()}
for a in alerts:
    rule = RULES.get(a["kibana.alert.rule.uuid"])
    monitor = a["monitor.name"] or ""
    if rule is None or (rule == "default") != ("baseline)" in monitor):
        continue
    start = datetime.fromisoformat(a["started"].replace("Z", "+00:00"))
    locations = a["locations"] if isinstance(a["locations"], list) else [a["locations"]]
    cause, delay = CAUSES[None], None
    candidates = [
        w for w in windows if w["start"] <= start <= w["end"] + timedelta(minutes=10)
    ]
    if candidates:
        w = max(candidates, key=lambda w: w["start"])
        if not (
            w["truth"] == "noise:single-location" and "office-network" not in locations
        ):
            cause, delay = CAUSES[w["truth"]], round(
                (start - w["start"]).total_seconds() / 60, 1
            )
    counts[rule][cause].append(
        {
            "start": f"{start:%H:%M:%S}",
            "monitor": monitor,
            "locations": ", ".join(sorted(locations)),
            "delay_min": "" if delay is None else f"+{delay}",
        }
    )

print(f"{'Cause':<32}{'Default rule':>14}{'Scoped rule':>13}")
for cause in CAUSES.values():
    print(
        f"{cause:<32}{len(counts['default'][cause]):>14}{len(counts['scoped'][cause]):>13}"
    )
print(
    f"{'Total':<32}{sum(map(len, counts['default'].values())):>14}{sum(map(len, counts['scoped'].values())):>13}"
)

for rule in ("default", "scoped"):
    print(f"\n{rule} rule alerts")
    rows = [
        {"cause": cause, **a} for cause, items in counts[rule].items() for a in items
    ]
    table(
        sorted(rows, key=lambda r: r["start"]),
        ["start", "cause", "monitor", "locations", "delay_min"],
    )

## 11. Read the SLO and the journey's checks

The SLO summary updates every few minutes: rerun the cell if an instance shows no data.

In [ ]:
instances = kbn(
    "GET",
    "/api/observability/slos",
    params={"kqlQuery": f'slo.name:"{SLO_NAME}"', "perPage": 25},
)["results"]
for s in instances:
    summary = s.get("summary", {})
    burn = summary.get("oneHourBurnRate") or summary.get("fiveMinuteBurnRate")
    print(
        f"{s.get('instanceId', '*'):<70} sli={summary.get('sliValue'):<8} "
        f"status={summary.get('status')}  1h burn rate={burn}"
    )

outages = [w for w in windows if w["truth"].startswith("incident")]
in_outage = " OR ".join(
    f'(@timestamp >= TO_DATETIME("{iso(w["start"])}") AND @timestamp < TO_DATETIME("{iso(w["end"])}"))'
    for w in outages
)
rows = esql(
    f"""
FROM synthetics-*
| WHERE monitor.project.id == "{PROJECT_ID}" AND monitor.type == "browser"
    AND summary.down IS NOT NULL AND summary.final_attempt == true
    AND @timestamp >= TO_DATETIME("{iso(T0)}") AND @timestamp < TO_DATETIME("{iso(END)}")
| EVAL in_outage = {in_outage}
| STATS checks = COUNT(*), down = COUNT(CASE(monitor.status == "down", 1, null)) BY in_outage, observer.geo.name
| SORT in_outage, observer.geo.name
"""
)
table(rows, ["in_outage", "observer.geo.name", "checks", "down"])

failed = sum(r["down"] for r in rows)
failed_outage = sum(r["down"] for r in rows if r["in_outage"])
outside = [r for r in rows if not r["in_outage"]]
outside_checks, outside_down = sum(r["checks"] for r in outside), sum(
    r["down"] for r in outside
)
print(
    f"\nfailed final checks: {failed} = {failed_outage} in the outages + {failed - failed_outage} outside"
)
print(
    f"outside the outages: {outside_down} of {outside_checks} checks down "
    f"({outside_down / max(outside_checks, 1):.1%})"
)
print(
    f"background alerts from the scoped rule: {len(counts['scoped']['Background 503s'])}"
)

## 12. Clean up

Deletes only what this notebook created and resets the app's chaos state. Delete Bean Box from your host yourself.

In [ ]:
def attempt(label, fn):
    try:
        fn()
        print(f"deleted {label}")
    except Exception as error:  # keep going so one failure doesn't block the rest
        print(f"skipped {label}: {error}")


for project in state.get("projects", [PROJECT_ID, BASELINE_PROJECT_ID]):
    for m in (
        project_monitors(project)
        if "project_monitors" in globals()
        else kbn(
            "GET",
            "/api/synthetics/monitors",
            params={"projects": project, "perPage": 100},
        ).get("monitors", [])
    ):
        monitor_id = m.get("config_id") or m["id"]
        attempt(
            f"monitor {m['name']}",
            lambda: kbn("DELETE", f"/api/synthetics/monitors/{monitor_id}", ok=(404,)),
        )

if state.get("rule_id"):
    attempt(
        f"rule {RULE_NAME}",
        lambda: kbn("DELETE", f"/api/alerting/rule/{state['rule_id']}", ok=(404,)),
    )
if state.get("default_rule_created") and state.get("default_rule_id"):
    attempt(
        "default status rule",
        lambda: kbn(
            "DELETE", f"/api/alerting/rule/{state['default_rule_id']}", ok=(404,)
        ),
    )
if state.get("slo_id"):
    attempt(
        f"SLO {SLO_NAME}",
        lambda: kbn("DELETE", f"/api/observability/slos/{state['slo_id']}", ok=(404,)),
    )
if state.get("maintenance_window_id"):
    attempt(
        f"maintenance window {MW_TITLE}",
        lambda: kbn(
            "DELETE",
            f"/api/maintenance_window/{state['maintenance_window_id']}",
            ok=(404,),
        ),
    )
if state.get("private_location_id"):
    attempt(
        f"Private Location {LOCATION_LABEL}",
        lambda: kbn(
            "DELETE",
            f"/api/synthetics/private_locations/{state['private_location_id']}",
            ok=(404,),
        ),
    )

attempt(
    f"container {AGENT_CONTAINER}",
    lambda: subprocess.run(
        ["docker", "rm", "-f", AGENT_CONTAINER], capture_output=True, check=True
    ),
)
if state.get("agent_policy_id"):
    policy_id = state["agent_policy_id"]
    attempt(
        "agents of the policy",
        lambda: kbn(
            "POST",
            "/api/fleet/agents/bulk_unenroll",
            json={
                "agents": f'policy_id:"{policy_id}"',
                "revoke": True,
                "force": True,
                "includeInactive": True,
            },
        ),
    )

    def delete_policy():
        # Unenrolling runs asynchronously, so retry until the policy has no agents left.
        for _ in range(12):
            try:
                kbn(
                    "POST",
                    "/api/fleet/agent_policies/delete",
                    json={"agentPolicyId": policy_id},
                    ok=(404,),
                )
                return
            except RuntimeError as error:
                last = error
                time.sleep(10)
        raise last

    attempt(f"agent policy {AGENT_POLICY_NAME}", delete_policy)

attempt("chaos state (reset)", lambda: chaos(CLEAR, "notebook: clean up"))
STATE_FILE.unlink(missing_ok=True)
print("done")